In [ ]:
import pandas as pd
import sqlite3

# Carrega os dados tratados
df = pd.read_csv("../data/processed/cobertura_vacinal_por_uf.csv")

# Cria (ou conecta a) um banco SQLite
conn = sqlite3.connect("../data/processed/cobertura_vacinal.db")

# Envia o DataFrame como uma tabela SQL
df.to_sql("cobertura_vacinal", conn, if_exists="replace", index=False)

print("Banco de dados criado com sucesso!")

In [ ]:
query_teste = "SELECT * FROM cobertura_vacinal LIMIT 5;"
pd.read_sql(query_teste, conn)

In [ ]:
query1 = """
SELECT
    uf,
    vacina,
    ano,
    cobertura,
    LAG(cobertura) OVER (PARTITION BY uf, vacina ORDER BY ano) AS cobertura_ano_anterior,
    ROUND(cobertura - LAG(cobertura) OVER (PARTITION BY uf, vacina ORDER BY ano), 2) AS variacao_absoluta
FROM cobertura_vacinal
WHERE uf = 'Roraima'
ORDER BY vacina, ano;
"""

pd.read_sql(query1, conn)

In [ ]:
query2 = """
WITH variacao_por_estado AS (
    SELECT
        uf,
        AVG(CASE WHEN ano = 2019 THEN cobertura END) AS cobertura_2019,
        AVG(CASE WHEN ano = 2022 THEN cobertura END) AS cobertura_2022
    FROM cobertura_vacinal
    WHERE ano IN (2019, 2022)
    GROUP BY uf
)
SELECT
    uf,
    ROUND(cobertura_2019, 2) AS cobertura_2019,
    ROUND(cobertura_2022, 2) AS cobertura_2022,
    ROUND(cobertura_2022 - cobertura_2019, 2) AS variacao,
    RANK() OVER (ORDER BY (cobertura_2022 - cobertura_2019) ASC) AS ranking_queda
FROM variacao_por_estado
ORDER BY ranking_queda
LIMIT 10;
"""

pd.read_sql(query2, conn)

In [ ]:
query3 = """
SELECT
    vacina,
    ano,
    ROUND(AVG(cobertura), 2) AS cobertura_media_nacional,
    ROUND(AVG(AVG(cobertura)) OVER (
        PARTITION BY vacina 
        ORDER BY ano 
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS media_movel_3_anos
FROM cobertura_vacinal
GROUP BY vacina, ano
ORDER BY vacina, ano;
"""

pd.read_sql(query3, conn)